# Context Managers

Decent introductions:

- [https://realpython.com/python-with-statement/](https://realpython.com/python-with-statement/)
- [https://medium.com/@sasidharan01/understanding-and-implementing-python-context-managers-8e45884dfe14](https://medium.com/@sasidharan01/understanding-and-implementing-python-context-managers-8e45884dfe14)
- [https://www.pythonmorsels.com/creating-a-context-manager/](https://www.pythonmorsels.com/creating-a-context-manager/)

Context managers are a way to manage resources in Python. Python’s with statement allows you to manage external resources safely by using objects that support the context manager protocol. These objects automatically handle the setup and cleanup phases of common operations. Context managers have in common that they set up some resource and then close it down after all work with the resource is done, even after something went wrong.

<img src="anatomy.png" width="300" style="margin-left:0;" />

In [ ]:
# Every time you open a file you use a context manager, albeit a simple one that just opens the file
# and then closes it after you exit the body of the with-clause.

with open('example.txt') as fh:
    for line in fh:
        print(line, end='')

In [ ]:
# The context optionally returns an object (the opened file in the case below) that can be
# accessed after the context is cleaned up

with open('example.txt') as f:
    data = f.readlines()

print(data)
f.closed

In [ ]:
# This is doing much of what the first example above did, but we are introducing some explicit
# exception handling. Under the hood, a context manager behaves a bit like a try-except-finally
# clause. 

try:
    print('--- entering...')
    fh = open('example.txt')
    for line in fh:
        print(line, end='')
    #1/0
    #x = y
    #raise Exception
except ZeroDivisionError as e:
    print('--- ignoring zero division error...')
except Exception as e:
    print('--- other error, not ignoring it...')
    raise e
finally:
    print('--- closing...')
    fh.close()

In [ ]:
# Let's use our own File class, it behaves similar to the first example above when there are no errors,
# but it also handles some exceptions similar to the if-then-else examples.

from utils import File

with File('example.txt') as fh:
    for line in fh:
        print(line, end='')

In [ ]:
# Now using the File class, but introducing an error in the body of the with-clause. The code will not
# crash and the file will still be closed.

with File('example.txt', debug=True) as fh:
    for line in fh:
        print(line, end='')
    1/0

### The File context manager

In [ ]:
# This is the code for the utils.File class that was used above.

class File:

    """For a file setting up is opening it and closing is... closing it. You use the
    dunder methods (aka magic methods) __enter__ method and __exit__ to do that. When
    you add those methods you have created a context manager."""

    def __init__(self, file_name: str, debug=False, method='r'):
        # For initialization we typically store some information relevant to the 
        # resource, like the file name for example.
        if debug:
            print('--- initializing...')
        self.debug = debug
        self.name = file_name
        self.method = method

    def __str__(self):
        return f'<File name="{self.name}'

    def __enter__(self):
        # This happens after initialization, we just open the file and save the
        # file handle, which we return.
        if self.debug:
            print('--- entering...')
        self.fh = open(self.name, self.method)
        return self.fh

    def __exit__(self, exception, message, traceback):
        # This will always run, unless an error occured in __init__ or __enter__.
        if self.debug:
            print('--- closing...')
        self.fh.close()
        # When cleaning up you can also deal with errors
        if exception is ZeroDivisionError:
            # dealing with the error and returning True
            print('--- ignoring zero division error...')
            return True
        if exception is FileNotFoundError:
            # We know about this error, but we don't want to trap it, returning False raises
            # the error again.
            print('--- file not found, exiting with an error...')
            print('---', exception, message)
            return False
        elif exception is not None:
            # And all the other possible problems, let's say we just want to alert is to them
            # but not break the script.
            print('--- other error, ignoring it, but reportinh...')
            print('---', exception, message)
            return True

In [ ]:
# The example above (repeated here) works seamless because the context manager handles the error.

with File('example.txt', debug=True) as fh:
    for line in fh:
        print(line, end='')
    1/0
print('--- continuing')

In [ ]:
# This exceptions are not ignored by the context manager, but the file is still closed.

with File('example.txt', debug=True) as fh:
    open('xxx')
print('--- continuing')

In [ ]:
# Other exceptions are trapped the context manager, but duly noted.

with File('example.txt', debug=True) as fh:
    print(unknown_variable)
print('--- continuing')

In [ ]:
# This one will still crash the script since the error doesn't occur in the body of the context
# and the File will not be closed.

with File('xxx') as fh:
    print('Howdy')
print('--- continuing')